In [22]:
from utilities_2 import preprocess, save_preds, evaluate_f1, evaluate_f1_classwise_all
import os
from huggingface_hub import from_pretrained_keras
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    AutoConfig,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification,
    pipeline
)


In [ ]:
! huggingface-cli login
# tutorial for huggingface login: https://youtu.be/s95A-au3vsM?si=kRSgBGUJDj_mlY4B

In [3]:
#Or
from huggingface_hub import interpreter_login

interpreter_login()


    _|    _|  _|    _|    _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|_|_|_|    _|_|      _|_|_|  _|_|_|_|
    _|    _|  _|    _|  _|        _|          _|    _|_|    _|  _|            _|        _|    _|  _|        _|
    _|_|_|_|  _|    _|  _|  _|_|  _|  _|_|    _|    _|  _|  _|  _|  _|_|      _|_|_|    _|_|_|_|  _|        _|_|_|
    _|    _|  _|    _|  _|    _|  _|    _|    _|    _|    _|_|  _|    _|      _|        _|    _|  _|        _|
    _|    _|    _|_|      _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|        _|    _|    _|_|_|  _|_|_|_|



In [ ]:
language_code="bg"

# Replace nateraw with your username that you logged in with!
model_name = "OOOss/bg-ner-model"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForTokenClassification.from_pretrained(model_name)


tokenizer_config.json:   0%|          | 0.00/1.18k [00:00<?, ?B/s]

c:\Users\user\anaconda3\envs\nlp_gpu\lib\site-packages\huggingface_hub\file_download.py:144: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\user\.cache\huggingface\hub\models--OOOss--bg-ner-model. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.09k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.24G [00:00<?, ?B/s]

In [ ]:
language_code="ru_no_lem"

# Replace nateraw with your username that you logged in with!
model_name = "OOOss/ru-not-lem-ner-model"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForTokenClassification.from_pretrained(model_name)

In [ ]:
language_code="ru"

# Replace nateraw with your username that you logged in with!
model_name = "MariiaZviahintseva/nlp_exp"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForTokenClassification.from_pretrained(model_name)

In [ ]:
language_code="bg+ru"
model_name = "tretiakt/bg-ru-ner-model"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForTokenClassification.from_pretrained(model_name)

model.safetensors:  66%|######5   | 1.47G/2.24G [00:00<?, ?B/s]

In [3]:
language_code="all"
model_name = "mihael199/slavic-ner-model"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForTokenClassification.from_pretrained(model_name)

tokenizer_config.json:   0%|          | 0.00/1.18k [00:00<?, ?B/s]

c:\Users\user\anaconda3\envs\nlp_gpu\lib\site-packages\huggingface_hub\file_download.py:144: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\user\.cache\huggingface\hub\models--mihael199--slavic-ner-model. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.09k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.24G [00:00<?, ?B/s]

In [5]:
from datasets import DatasetDict,Dataset, concatenate_datasets
from collections import defaultdict
#import stanza
import pandas as pd
import re, os
import numpy as np
import evaluate
import torch.nn as nn
from transformers import XLMRobertaConfig
from transformers.modeling_outputs import TokenClassifierOutput
from transformers.models.roberta.modeling_roberta import RobertaModel
from transformers.models.roberta.modeling_roberta import RobertaPreTrainedModel
from seqeval.metrics import f1_score
from torch.utils.data import DataLoader
from transformers import EvalPrediction
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    AutoConfig,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification
)

def parse_iob2(file_path):
    sentences, labels = [], []
    words, tags = [], []
    unique_labels = set()

    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("#"):  # Skip metadata lines
                if words:
                    sentences.append(words)
                    labels.append(tags)
                    words, tags = [], []
                continue
            parts = line.split()  # Split by whitespace
            if len(parts) >= 3:
                words.append(parts[1])  # First column: token
                tags.append(parts[2])  # Second column: label
                unique_labels.add(parts[2])
        if words:
            sentences.append(words)
            labels.append(tags)

    return {"tokens": sentences, "ner_tags": labels}, sorted(unique_labels)

def generate_data_files(language_code: str) -> dict:
    return {
        "train": f"data/train/lemmatized_{language_code}.txt",
        "validation": f"data/val/lemmatized_{language_code}.txt",
        "test": f"data/test/{language_code}.txt"
    }

# Convert labels to integers
def convert_labels(dataset,label2id):
    dataset["ner_tags"] = [[label2id[tag] for tag in tags] for tags in dataset["ner_tags"]]
    return dataset

def load_data(language_code: str, train=True):
    
    data_files = generate_data_files(language_code)
    dataset_dict = {}
    all_labels = set()
    #for _,file in data_files.items():
    if train==True:
      splits=["train","validation"]
    elif train=="all":
        splits=["train","validation","test"]
    else:
      splits=["test"]
    for split in splits:
      parsed_data, labels = parse_iob2(data_files[split])
      dataset_dict[split] = parsed_data
      all_labels.update(labels)

    # Convert label list to mapping
    label_list = sorted(all_labels)  # Ensure consistent order
    label2id = {label: i for i, label in enumerate(label_list)}
    id2label = {i: label for label, i in label2id.items()}

    dataset_dict = {split: convert_labels(dataset , label2id) for split, dataset in dataset_dict.items()}
    raw_datasets = DatasetDict({
        split: Dataset.from_dict(dataset_dict[split]) for split in dataset_dict
    })

    return raw_datasets, label_list, label2id, id2label

def tokenize_and_align_labels(examples, tokenizer):
    tokenized_inputs = tokenizer(
        examples["tokens"],
        is_split_into_words=True,
        padding="max_length",
        truncation=True,
        max_length=128,
    )

    all_labels = []
    for i, label in enumerate(examples["ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:
            if word_idx is None or word_idx == previous_word_idx:
                label_ids.append(-100)
            else:
                label_ids.append(label[word_idx])
            previous_word_idx = word_idx
        all_labels.append(label_ids)

    tokenized_inputs["labels"] = all_labels
    return tokenized_inputs


def preprocess_1(language_code: str, model_name: str, train:bool, cyrillic: bool = False):

    # Lemmatize
    #for split in ["train", "validation", "test"]:
        #original = f"data/{split}/{language_code}.txt"
        #lemmatized = f"data/{split}/lemmatized_{language_code}.txt"
        #lemmatize_file(original, lemmatized, language_code)

    # Optional transliteration
    #if language_code == "sl":
    #    for split in ["train", "validation", "test"]:
    #        lemma_file = f"data/{split}/lemmatized_{language_code}.txt"
    #        translit_file = f"data/{split}/{language_code}.txt"
    #        transliterate_file(lemma_file, translit_file)
    #else:
    #    for split in ["train", "validation", "test"]:
    #        lemma_file = f"data/{split}/lemmatized_{language_code}.txt"
    #        final_file = f"data/{split}/{language_code}.txt"
    #        import shutil
    #        shutil.copyfile(lemma_file, final_file)

    # Load dataset
    if language_code=="sl" and cyrillic:
       language_code+="_cyrilic"
    raw_datasets, label_list, label2id, id2label = load_data(language_code,train)

    # Tokenize
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    print(raw_datasets["test"][0]["tokens"])
    tokenized_datasets = encode_dataset(raw_datasets, tokenizer)
    
    return tokenized_datasets, label_list, label2id, id2label, tokenizer

def encode_dataset(corpus, tokenizer):
    return corpus.map(lambda x: tokenize_and_align_labels(x, tokenizer), batched=True, 
                      remove_columns=[ 'ner_tags', 'tokens'])

def generate_list_for_compute_metrics(predictions, label_ids, index2tag):
    """
    Function to generate prediction and true labels lists for computing metrics.

    Parameters:
    predictions (np.ndarray): A 2D numpy array containing the predicted label IDs for each token in each example.
    label_ids (np.ndarray): A 2D numpy array containing the true label IDs for each token in each example.

    Returns:
    preds_labels_list (list): A list of lists, where each sublist contains the predicted labels for each token in an example.
    true_labels_list (list): A list of lists, where each sublist contains the true labels for each token in an example.
    """
    # Get the predicted labels by taking the argmax over the second dimension of the predictions array
    preds = np.argmax(predictions, axis=2)
    batch_size, seq_len = preds.shape
    preds_labels_list, true_labels_list = [], []

    # Iterate over each example in the batch
    for batch_idx in range(batch_size):
        example_labels, example_preds = [], []
        # Iterate over each token in the example
        for seq_idx in range(seq_len):
            # Ignore tokens with label ID = -100 (these are special tokens or subwords that we masked during training)
            if label_ids[batch_idx, seq_idx] != -100:
                # Append the predicted and true labels for the token to the lists for the current example
                example_preds.append(index2tag[preds[batch_idx][seq_idx]])
                example_labels.append(index2tag[label_ids[batch_idx][seq_idx]])
        # Append the lists for the current example to the main lists
        preds_labels_list.append(example_preds)
        true_labels_list.append(example_labels)

    # Return the lists of predicted and true labels
    return preds_labels_list, true_labels_list

def save_preds_1(model, language_code, target_language, tokenized_dataset, tokenizer):
    
    decoded_sentences = [
    tokenizer.convert_tokens_to_string(tokenizer.convert_ids_to_tokens(input_ids, skip_special_tokens=True)).split()
    for input_ids in tokenized_dataset["input_ids"]
    ]
    # Assuming your tokenizer is already defined
    data_collator = DataCollatorForTokenClassification(tokenizer, padding=True)
    
    # Make sure your dataset is tokenized and has correct features
    test_loader = DataLoader(
        tokenized_dataset,  # should be Hugging Face Dataset
        batch_size=16,
        collate_fn=data_collator
    )

    # Device setup
    if torch.cuda.is_available():
        device = torch.device("cuda")
        print(f"CUDA is available. Using GPU: {torch.cuda.get_device_name(0)}")
    else:
        device = torch.device("cpu")
        print("CUDA not available. Using CPU.")
    
    # Move model to device
    model.to(device)
    model.eval()

    all_predictions = []

    with torch.no_grad():
        for batch in test_loader:
            # Move input tensors to the same device as the model
            batch = {k: v.to(device) for k, v in batch.items() if k in ['input_ids', 'attention_mask']}
            
            outputs = model(**batch)
            preds = torch.argmax(outputs.logits, dim=-1)
            
            all_predictions.append(preds.cpu().numpy())  # convert to CPU numpy for saving
    
    label_ids = np.array(tokenized_dataset["labels"])
    # Convert logits to labels
    #predictions = torch.argmax(outputs.logits, dim=-1)
    id2label = model.config.id2label
    all_predictions=np.concatenate(all_predictions, axis=0)
    batch_size, seq_len = all_predictions.shape
    preds_labels_list = []

    # Iterate over each example in the batch
    for batch_idx in range(batch_size):
        sentence_iob = []
        token_idx=0
        # Iterate over each token in the example

        for seq_idx in range(seq_len):
            if token_idx >= len(decoded_sentences[batch_idx]):
                break  # all real tokens processed, exit early!

            if label_ids[batch_idx][seq_idx] != -100:
                token = decoded_sentences[batch_idx][token_idx]
                pred_label = id2label[all_predictions[batch_idx][seq_idx]]
                token_idx += 1
                sentence_iob.append(f"{token_idx}\t{token}\t{pred_label}")
        # Append the lists for the current example to the main lists
        preds_labels_list.append(sentence_iob)

    os.makedirs(f"data/preds/{language_code}", exist_ok=True)
    with open(f"data/preds/{language_code}/{target_language}_predictions.iob", "w") as f:
        for sentence in preds_labels_list:
            for line in sentence:
                f.write(f"{line}\n")
            f.write("\n")  # Separate sentences with a blank line


In [25]:
os.makedirs("data/preds", exist_ok=True)

lang_list=["sl_cyrilic_non_lem"]
for target_language in lang_list:
  
  if target_language!="sl_cyrilic":
    target_lang_data=preprocess_1(language_code=target_language, model_name=model_name, train=False)
  else:
    target_lang_data=preprocess_1(language_code=target_language, model_name=model_name, train=False, cyrillic=True)
  
  tokenized_test, id2label, tokenizer = target_lang_data[0]["test"], target_lang_data[3], target_lang_data[4]
  save_preds_1(model, language_code, target_language, tokenized_dataset=tokenized_test, tokenizer=tokenizer)
  print(target_language + " done")

['Сооченье', ',', 'какршнега', 'ше', 'ни', 'било', '.']


Map:   0%|          | 0/4386 [00:00<?, ? examples/s]

CUDA is available. Using GPU: NVIDIA GeForce RTX 3050 Laptop GPU
sl_cyrilic_non_lem done


In [ ]:
os.makedirs("data/preds", exist_ok=True)

lang_list=["bg", "ru", "uk", "sl", "sl_cyrilic"]
for target_language in lang_list:
  
  if target_language!="sl_cyrilic":
    target_lang_data=preprocess(language_code=target_language, model_name=model_name, train=False)
  else:
    target_lang_data=preprocess(language_code=target_language, model_name=model_name, train=False, cyrillic=True)
  
  tokenized_test, id2label, tokenizer = target_lang_data[0]["test"], target_lang_data[3], target_lang_data[4]
  save_preds(model, language_code, target_language, tokenized_dataset=tokenized_test, tokenizer=tokenizer)
  print(target_language + " done")

Map:   0%|          | 0/3384 [00:00<?, ? examples/s]

CUDA is available. Using GPU: NVIDIA GeForce RTX 3050 Laptop GPU
bg done


Map:   0%|          | 0/10486 [00:00<?, ? examples/s]

CUDA is available. Using GPU: NVIDIA GeForce RTX 3050 Laptop GPU
ru done


Map:   0%|          | 0/1493 [00:00<?, ? examples/s]

CUDA is available. Using GPU: NVIDIA GeForce RTX 3050 Laptop GPU
uk done


Map:   0%|          | 0/4312 [00:00<?, ? examples/s]

CUDA is available. Using GPU: NVIDIA GeForce RTX 3050 Laptop GPU
sl done


Map:   0%|          | 0/4312 [00:00<?, ? examples/s]

CUDA is available. Using GPU: NVIDIA GeForce RTX 3050 Laptop GPU
sl_cyrilic done


In [8]:
import torch
# Function to free GPU memory
def free_gpu_memory(model):
    import gc
    del model
    gc.collect()
    torch.cuda.empty_cache()

In [6]:
tokenized_test1, id2label = target_lang_data[0]["test"]["input_ids"][0], target_lang_data[3]
tokenized_test2 = target_lang_data[0]["test"]["labels"][0]
print(id2label)

print(tokenized_test1)
    
print(tokenized_test2)
print(len(tokenized_test1), len(tokenized_test2))


{0: 'B-EVT', 1: 'B-LOC', 2: 'B-ORG', 3: 'B-PER', 4: 'B-PRO', 5: 'I-EVT', 6: 'I-LOC', 7: 'I-ORG', 8: 'I-PER', 9: 'I-PRO', 10: 'O'}
[0, 86956, 1091, 7553, 42803, 303, 50084, 29, 156467, 61, 152023, 12164, 6, 5, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
[-100, 10, 10, 10, 10, -100, 10, 10, 3, 10, 1, 6, 10, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100,

## Test bg model

In [8]:
print("bg_model_on_bg_test")
gold = 'data/test/lemmatized_bg.txt'
pred = 'data/preds/bg/bg_predictions.iob'
evaluate_f1(gold, pred)
print("\nbg_model_on_ru_test")
gold = 'data/test/lemmatized_ru.txt'
pred = 'data/preds/bg/ru_predictions.iob'
evaluate_f1(gold, pred)
print("\nbg_model_on_uk_test")
gold = 'data/test/lemmatized_uk.txt'
pred = 'data/preds/bg/uk_predictions.iob'
evaluate_f1(gold, pred)
print("\nbg_model_on_sl_test")
gold = 'data/test/lemmatized_sl.txt'
pred = 'data/preds/bg/sl_predictions.iob'
evaluate_f1(gold, pred)
print("\nbg_model_on_slt_test")
gold = 'data/test/lemmatized_sl_cyrilic.txt'
pred = 'data/preds/bg/sl_cyrilic_predictions.iob'
evaluate_f1(gold, pred)

bg_model_on_bg_test
recall:    0.8153003650846332
precision: 0.9331433998100664
slot-f1:   0.8702506421043309

unlabeled
ul_recall:    0.833388649186857
ul_precision: 0.9538461538461539
ul_slot-f1:   0.8895580550881232

loose (partial overlap with same label)
l_recall:    0.8411881845336874
l_precision: 0.9477682811016145
l_slot-f1:   0.8913033883754764

bg_model_on_ru_test
recall:    0.6874018038987489
precision: 0.8359043305972261
slot-f1:   0.7544145352364531

unlabeled
ul_recall:    0.7427407622926971
ul_precision: 0.9031984149448061
ul_slot-f1:   0.8151483219976371

loose (partial overlap with same label)
l_recall:    0.7110852487634565
l_precision: 0.8630767053495613
l_slot-f1:   0.7797432940389497

bg_model_on_uk_test
recall:    0.7123107307439105
precision: 0.8466353677621283
slot-f1:   0.7736860922416875

unlabeled
ul_recall:    0.7557603686635944
ul_precision: 0.8982785602503912
ul_slot-f1:   0.8208795137647479

loose (partial overlap with same label)
l_recall:    0.728110599

## Test ru model

In [15]:
print("ru_model_on_bg_test")
gold = 'data/test/lemmatized_bg.txt'
pred = 'data/preds/ru/bg_predictions.iob'
evaluate_f1(gold, pred)
print("\nru_model_on_ru_test")
gold = 'data/test/lemmatized_ru.txt'
pred = 'data/preds/ru/ru_predictions.iob'
evaluate_f1(gold, pred)
print("\nru_model_on_uk_test")
gold = 'data/test/lemmatized_uk.txt'
pred = 'data/preds/ru/uk_predictions.iob'
evaluate_f1(gold, pred)
print("\nru_model_on_sl_test")
gold = 'data/test/lemmatized_sl.txt'
pred = 'data/preds/ru/sl_predictions.iob'
evaluate_f1(gold, pred)
print("\nru_model_on_slt_test")
gold = 'data/test/lemmatized_sl_cyrilic.txt'
pred = 'data/preds/ru/sl_cyrilic_predictions.iob'
evaluate_f1(gold, pred)

ru_model_on_bg_test
recall:    0.7965482907401261
precision: 0.9165552797403094
slot-f1:   0.8523483974074403

unlabeled
ul_recall:    0.82625290408231
ul_precision: 0.9507351537139583
ul_slot-f1:   0.884133889727426

loose (partial overlap with same label)
l_recall:    0.8265847992034517
l_precision: 0.9375596715676914
l_slot-f1:   0.8785817552972611

ru_model_on_ru_test
recall:    0.7124236252545825
precision: 0.8637037037037038
slot-f1:   0.7808035714285714

unlabeled
ul_recall:    0.7568228105906314
ul_precision: 0.9175308641975308
ul_slot-f1:   0.8294642857142858

loose (partial overlap with same label)
l_recall:    0.7338376491125982
l_precision: 0.8848677248677249
l_slot-f1:   0.8023069070263531

ru_model_on_uk_test
recall:    0.771560236998025
precision: 0.886535552193646
slot-f1:   0.8250615980288631

unlabeled
ul_recall:    0.804476629361422
ul_precision: 0.924357034795764
ul_slot-f1:   0.8602604716649067

loose (partial overlap with same label)
l_recall:    0.795260039499670

## Test bg+ru model

In [8]:
print("bg+ru_model_on_bg_test")
gold = 'data/test/lemmatized_bg.txt'
pred = 'data/preds/bg+ru/bg_predictions.iob'
evaluate_f1(gold, pred)
print("\nbg+ru_model_on_ru_test")
gold = 'data/test/lemmatized_ru.txt'
pred = 'data/preds/bg+ru/ru_predictions.iob'
evaluate_f1(gold, pred)
print("\nbg+ru_model_on_uk_test")
gold = 'data/test/lemmatized_uk.txt'
pred = 'data/preds/bg+ru/uk_predictions.iob'
evaluate_f1(gold, pred)
print("\nbg+ru_model_on_sl_test")
gold = 'data/test/lemmatized_sl.txt'
pred = 'data/preds/bg+ru/sl_predictions.iob'
evaluate_f1(gold, pred)
print("\nbg+ru_model_on_slt_test")
gold = 'data/test/lemmatized_sl_cyrilic.txt'
pred = 'data/preds/bg+ru/sl_cyrilic_predictions.iob'
evaluate_f1(gold, pred)

bg+ru_model_on_bg_test
recall:    0.8123133089943578
precision: 0.9242824773413897
slot-f1:   0.8646882176293942

unlabeled
ul_recall:    0.8373713906405575
ul_precision: 0.9527945619335347
ul_slot-f1:   0.8913619501854796

loose (partial overlap with same label)
l_recall:    0.8372054430799867
l_precision: 0.9382552870090635
l_slot-f1:   0.8848547534398745

bg+ru_model_on_ru_test
recall:    0.7152167588012802
precision: 0.8575315705016395
slot-f1:   0.7799352750809061

unlabeled
ul_recall:    0.763223741635147
ul_precision: 0.9150910486290379
ul_slot-f1:   0.8322863125832858

loose (partial overlap with same label)
l_recall:    0.7365144020948502
l_precision: 0.877764599176725
l_slot-f1:   0.8009597701926757

bg+ru_model_on_uk_test
recall:    0.760697827518104
precision: 0.868796992481203
slot-f1:   0.8111618111618112

unlabeled
ul_recall:    0.8008558262014484
ul_precision: 0.9146616541353384
ul_slot-f1:   0.853983853983854

loose (partial overlap with same label)
l_recall:    0.7850

## Test on all_langs model

In [6]:
print("all_model_on_bg_test")
gold = 'data/test/lemmatized_bg.txt'
pred = 'data/preds/all/bg_predictions.iob'
evaluate_f1(gold, pred)
print("\nall_model_on_ru_test")
gold = 'data/test/lemmatized_ru.txt'
pred = 'data/preds/all/ru_predictions.iob'
evaluate_f1(gold, pred)
print("\nall_model_on_uk_test")
gold = 'data/test/lemmatized_uk.txt'
pred = 'data/preds/all/uk_predictions.iob'
evaluate_f1(gold, pred)
print("\nall_model_on_sl_test")
gold = 'data/test/lemmatized_sl.txt'
pred = 'data/preds/all/sl_predictions.iob'
evaluate_f1(gold, pred)
print("\nall_model_on_slt_test")
gold = 'data/test/lemmatized_sl_cyrilic.txt'
pred = 'data/preds/all/sl_cyrilic_predictions.iob'
evaluate_f1(gold, pred)

all_model_on_bg_test
recall:    0.8469963491536674
precision: 0.9366856303908974
slot-f1:   0.8895860566448801

unlabeled
ul_recall:    0.8655824759376037
ul_precision: 0.957239860524867
ul_slot-f1:   0.9091067538126362

loose (partial overlap with same label)
l_recall:    0.8670760039827414
l_precision: 0.9478803450174343
l_slot-f1:   0.9056794145647186

all_model_on_ru_test
recall:    0.7147512365434973
precision: 0.847103448275862
slot-f1:   0.7753195518384093

unlabeled
ul_recall:    0.7785859761419843
ul_precision: 0.9227586206896552
ul_slot-f1:   0.8445636736626164

loose (partial overlap with same label)
l_recall:    0.7345941227814955
l_precision: 0.867448275862069
l_slot-f1:   0.7955125355043637

all_model_on_uk_test
recall:    0.7472021066491112
precision: 0.8888018794048551
slot-f1:   0.8118741058655221

unlabeled
ul_recall:    0.7755102040816326
ul_precision: 0.9224745497259201
ul_slot-f1:   0.8426323319027181

loose (partial overlap with same label)
l_recall:    0.76596445

## Test on ru_no_lem_model model

In [47]:
print("ru_no_lem_model_on_bg_test")
gold = 'data/test/lemmatized_bg.txt'
pred = 'data/preds/ru_no_lem/bg_predictions.iob'
#evaluate_f1(gold, pred)
print("\nru_no_lem_model_on_ru_test")
gold = 'data/test/lemmatized_ru.txt'
pred = 'data/preds/ru_no_lem/ru_predictions.iob'
#evaluate_f1(gold, pred)
print("\nru_no_lem_model_on_uk_test")
gold = 'data/test/lemmatized_uk.txt'
pred = 'data/preds/ru_no_lem/uk_predictions.iob'
#evaluate_f1(gold, pred)
print("\nru_no_lem_model_on_sl_test")
gold = 'data/test/lemmatized_sl.txt'
pred = 'data/preds/ru_no_lem/sl_predictions.iob'
evaluate_f1(gold, pred)
print("\nru_no_lem_model_on_slt_test")
gold = 'data/test/lemmatized_sl_cyrilic_non_lem.txt'
pred = 'data/preds/ru_no_lem/sl_cyrilic_non_lem_predictions.iob'
evaluate_f1(gold, pred)

ru_no_lem_model_on_bg_test

ru_no_lem_model_on_ru_test

ru_no_lem_model_on_uk_test

ru_no_lem_model_on_sl_test
recall:    0.85866780529462
precision: 0.9202257474069555
slot-f1:   0.8883816816374613

unlabeled
ul_recall:    0.8869911756333618
ul_precision: 0.9505796217205613
ul_slot-f1:   0.9176851715505816

loose (partial overlap with same label)
l_recall:    0.874466268146883
l_precision: 0.9354789505796217
l_slot-f1:   0.9039442502231183

ru_no_lem_model_on_slt_test
recall:    0.02193225985563576
precision: 0.013736741436271953
slot-f1:   0.01689297551587726

unlabeled
ul_recall:    0.036368684064408664
ul_precision: 0.02277864719179273
ul_slot-f1:   0.028012402437720522

loose (partial overlap with same label)
l_recall:    0.07912270960577457
l_precision: 0.04938271604938271
l_slot-f1:   0.060811351452271095
